# **Graficas para redacción de articulo**
---
## *Subcoordinación de Posgrado y Educación Continua.*
### [Instituto Mexicano de Tecnología del Agua](https://www.gob.mx/imta).<br>

<img src="./Datos/Imagenes/Logos.png" style="height: 7em; vertical-align: middle;">

**Alumno: Ing. Omar Ulises Robles Pereyra** <br>
**Tutor: Dr. Ariosto Aguilar Chávez** <br>

[![Open In Colab](./Datos/Imagenes/colab-badge.svg)](https://colab.research.google.com/github/OmarURP/Toolbox_publica/blob/main/05_Graficacion.ipynb)

---

In [7]:
import ImtaTURB as imta
import numpy as np

In [8]:
# Datos numéricos
file_openfoam = "Datos/OpenFOAM/40_cm/U_Articulo"

# Carga de velocidades
U_probes, tiempo, coords, frecuencia = imta.cargar_U_OpenFOAM(file_openfoam)

# Nombre del archivo de salida
nombre_archivo = 'resumen_espectros_40cm.csv'

# Ventana de tiempo a analizar (Arranque en caliente simulación)
arranque = 50.00

# Componentes a guardar
componentes = ['r11', 'r12', 'r13', 'r22', 'r23', 'r33']

print(f"Iniciando procesamiento de probes... Guardando en: {nombre_archivo}\n")

with open(nombre_archivo, 'w') as f:
    # 1. Escribir encabezado
    headers = ["Probe"]
    for comp in componentes:
        headers.append(f"{comp}_frec")
        headers.append(f"{comp}_ener")
    
    header_str = ",".join(headers)
    f.write(header_str + "\n")
    print(header_str) # Imprimir encabezado para referencia

    # 2. Iterar sobre todos los probes (ordenados)
    for probe in sorted(U_probes.keys()):
        
        # --- Procesamiento de datos ---
        # Recortar tiempo (Eliminar el arranque en caliente)
        u1, u2, u3, tiempo_rec, _ = imta.recortar_tiempo(U_probes, probe, inicio=arranque)
        
        # Calcular fluctuantes
        u1_fluc, _ = imta.fluctuante(u1, tiempo=tiempo_rec, plot=False)
        u2_fluc, _ = imta.fluctuante(u2, tiempo=tiempo_rec, plot=False)
        u3_fluc, _ = imta.fluctuante(u3, tiempo=tiempo_rec, plot=False)
        
        # Calcular Autocorrelación
        tensor_r, _, _ = imta.autocorrelacion_norm(
            u1_fluc['fluc'], u2_fluc['fluc'], u3_fluc['fluc'], frecuencia, plot=False
        )
        
        # Calcular Espectros
        frecs_pos, espectro_dict, _ = imta.espectros_tensor(tensor_r, frecuencia, plot=False)
        
        # Construir la línea de texto
        row = [str(probe)]
        
        for comp in componentes:
            f_arr = frecs_pos[comp]
            e_arr = espectro_dict[comp]
            
            # Máximo
            idx_max = np.argmax(e_arr)
            val_frec = f_arr[idx_max]
            val_ener = e_arr[idx_max]
            
            # Formato: 3 decimales para frecuencia, notación científica para energía
            row.append(f"{val_frec:.4f}")
            row.append(f"{val_ener:.4e}")
        
        linea_completa = ",".join(row)
        
        # Guardar en CSV
        f.write(linea_completa + "\n")
        
        # Impresion en pantalla
        print(linea_completa)

print(f"\nArchivo {nombre_archivo} generado")

Se extrajeron 21 probes.
Tiempo inicial: 0.00 s, tiempo final: 200.00 s
Duración de la muestra: 200.00 s
Frecuencia de muestreo: 100.00 Hz (Δt = 0.0100 s)
Iniciando procesamiento de probes... Guardando en: resumen_espectros_40cm.csv

Probe,r11_frec,r11_ener,r12_frec,r12_ener,r13_frec,r13_ener,r22_frec,r22_ener,r23_frec,r23_ener,r33_frec,r33_ener
Componente r11: Frecuencia máxima = 0.733 Hz, Energía = 4.532e+06
Componente r12: Frecuencia máxima = 0.733 Hz, Energía = 6.781e+06
Componente r13: Frecuencia máxima = 0.080 Hz, Energía = 1.531e+05
Componente r22: Frecuencia máxima = 0.733 Hz, Energía = 1.015e+07
Componente r23: Frecuencia máxima = 0.733 Hz, Energía = 5.863e+03
Componente r33: Frecuencia máxima = 0.013 Hz, Energía = 1.082e+06
0,0.7333,4.5316e+06,0.7333,6.7809e+06,0.0800,1.5308e+05,0.7333,1.0146e+07,0.7333,5.8628e+03,0.0133,1.0817e+06
Componente r11: Frecuencia máxima = 0.733 Hz, Energía = 4.325e+06
Componente r12: Frecuencia máxima = 0.733 Hz, Energía = 6.629e+06
Componente r13